# GravLens-Net -- Phase 5: Matched Negative Sample

**Goal:** Phase 4 found the model was very likely separating "massive
elliptical" from "everything else" rather than detecting lensing arcs,
because SLACS positives are all massive ellipticals while the negatives
were random sky/random galaxies. Phase 5 fixes the negative sampling: real
LRG-type galaxies (fetched via a curated VizieR LRG-selection catalog),
with anything within 10 arcsec of a confirmed SLACS lens excluded.

**Short version of what's below:** this fixes the confound Phase 4 found,
but surfaces a second, subtler one -- still worth reading before trusting
the headline number.


In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              confusion_matrix, roc_auc_score)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, models

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 131 real SLACS positives (from Phase 4) + 422 real matched LRG negatives (Phase 5 fetch)
images = np.load("gravlens_phase5_images.npy")
labels = np.load("gravlens_phase5_labels.npy").astype(np.int32)
print(f"Phase 5 dataset: {images.shape}, positives={labels.sum()}/{len(labels)} "
      f"({100*labels.sum()/len(labels):.1f}%)")


I0000 00:00:1784471691.231116     797 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784471691.231696     797 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1784471691.281474     797 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1784471692.381232     797 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784471692.381700     797 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Phase 5 dataset: (553, 101, 101), positives=131/553 (23.7%)


## 1. Re-check the Phase 4 confound: concentration

Phase 4 found positives ~9x more centrally concentrated than negatives.
If the matched sample actually fixed this, that gap should have closed
(or even reversed, since LRGs are themselves centrally concentrated
galaxies).


In [2]:
def concentration(imgs, r_in=5, r_out=25):
    c = imgs.shape[1] // 2
    inner = imgs[:, c-r_in:c+r_in, c-r_in:c+r_in].mean(axis=(1, 2))
    outer = imgs[:, c-r_out:c+r_out, c-r_out:c+r_out].mean(axis=(1, 2))
    return inner / (outer + 1e-6)

pos_imgs = images[labels == 1]
neg_imgs = images[labels == 0]

conc_pos = concentration(pos_imgs)
conc_neg = concentration(neg_imgs)
print(f"Concentration -- positives (SLACS lenses): mean={conc_pos.mean():.2f}, std={conc_pos.std():.2f}")
print(f"Concentration -- negatives (matched LRGs): mean={conc_neg.mean():.2f}, std={conc_neg.std():.2f}")

conc_all = concentration(images)
conc_auc = roc_auc_score(labels, conc_all)
print(f"\nAUC using ONLY concentration as the score: {conc_auc:.3f} (0.5 = no signal)")


Concentration -- positives (SLACS lenses): mean=9.88, std=1.79
Concentration -- negatives (matched LRGs): mean=14.18, std=8.36

AUC using ONLY concentration as the score: 0.318 (0.5 = no signal)


**The Phase 4 confound is fixed.** Concentration AUC is now 0.32 (far
from the 1.0 that would indicate a usable shortcut) -- negatives are if
anything *more* concentrated than positives now, the opposite of Phase 4's
problem. Good sign. But that doesn't mean the dataset is clean -- checking
two more obvious shortcuts before trusting anything.


In [3]:
def total_flux(imgs):
    return imgs.sum(axis=(1, 2))

def half_light_radius_proxy(imgs):
    c = imgs.shape[1] // 2
    y, x = np.mgrid[0:imgs.shape[1], 0:imgs.shape[2]]
    r = np.sqrt((x - c) ** 2 + (y - c) ** 2)
    order = np.argsort(r.ravel())
    r_sorted = r.ravel()[order]
    radii = []
    for img in imgs:
        img_pos = np.clip(img, 0, None)
        total = img_pos.sum()
        if total <= 0:
            radii.append(np.nan)
            continue
        cumsum = np.cumsum(img_pos.ravel()[order])
        idx = np.searchsorted(cumsum, 0.5 * total)
        radii.append(r_sorted[idx])
    return np.array(radii)

flux_auc = roc_auc_score(labels, total_flux(images))
hlr_all = half_light_radius_proxy(images)
valid = ~np.isnan(hlr_all)
hlr_auc = roc_auc_score(labels[valid], hlr_all[valid])

print(f"AUC using ONLY total flux:             {flux_auc:.3f}")
print(f"AUC using ONLY half-light-radius proxy: {hlr_auc:.3f}")
print(f"\nMean total flux    -- pos: {total_flux(pos_imgs).mean():.1f}   neg: {total_flux(neg_imgs).mean():.1f}")


AUC using ONLY total flux:             0.790
AUC using ONLY half-light-radius proxy: 0.319

Mean total flux    -- pos: 260.5   neg: 404.0


**A second, subtler confound.** Total flux alone gets AUC 0.79 --
negatives (matched LRGs) tend to be brighter/larger than the SLACS
positives. This most likely reflects a redshift/selection mismatch: SLACS
targets a specific spectroscopic sample at a particular redshift range,
while this broader LRG catalog wasn't matched to that exact selection, only
to "massive elliptical-type galaxy" in general. Real LRGs at lower redshift
appear bigger and brighter simply from being closer.

This is smaller than Phase 4's confound (0.79 vs. Phase 4's much more
extreme concentration gap) but it's real, and it means part of the model's
score could still come from "is this a large/bright galaxy" rather than
"is there a lensing arc."


## 2. Train the CNN on this improved dataset

Same recipe as Phase 3b/4: standardize, augment positives (rotate/flip),
BatchNorm + GlobalAveragePooling CNN, ROC-AUC primary metric.


In [4]:
def standardize(imgs):
    mean = imgs.mean(axis=(1, 2), keepdims=True)
    std = imgs.std(axis=(1, 2), keepdims=True) + 1e-6
    return (imgs - mean) / std

def augment_positives(X_pos, rng, factor=6):
    out = []
    for img in X_pos:
        out.append(img)
        for _ in range(factor - 1):
            k = rng.integers(0, 4)
            aug = np.rot90(img, k)
            if rng.random() < 0.5:
                aug = np.fliplr(aug)
            out.append(aug)
    return np.array(out)

def build_cnn(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(16, 3, padding="same"), layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(32, 3, padding="same"), layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding="same"), layers.BatchNormalization(), layers.Activation("relu"),
        layers.GlobalAveragePooling2D(),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
                           tf.keras.metrics.Recall(name="recall"),
                           tf.keras.metrics.AUC(name="auc")])
    return model

images_std = standardize(images)
X_train, X_temp, y_train, y_temp = train_test_split(
    images_std, labels, test_size=0.30, stratify=labels, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=SEED)
print(f"train={len(y_train)} (pos={y_train.sum()})  val={len(y_val)} (pos={y_val.sum()})  test={len(y_test)} (pos={y_test.sum()})")

rng = np.random.default_rng(SEED)
pos_mask = y_train == 1
X_pos_aug = augment_positives(X_train[pos_mask], rng, factor=6)
X_train_aug = np.concatenate([X_train[~pos_mask], X_pos_aug], axis=0)
y_train_aug = np.concatenate([y_train[~pos_mask], np.ones(len(X_pos_aug), dtype=np.int32)])
perm = rng.permutation(len(X_train_aug))
X_train_aug, y_train_aug = X_train_aug[perm], y_train_aug[perm]

X_train_aug = X_train_aug[..., np.newaxis]
X_val_in = X_val[..., np.newaxis]
X_test_in = X_test[..., np.newaxis]

cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train_aug)
class_weight = {0: cw[0], 1: cw[1]}

model = build_cnn((images.shape[1], images.shape[2], 1))
early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True)
model.fit(X_train_aug, y_train_aug, validation_data=(X_val_in, y_val),
          epochs=20, batch_size=32, class_weight=class_weight, verbose=2, callbacks=[early_stop])


train=387 (pos=92)  val=83 (pos=20)  test=83 (pos=19)


Epoch 1/20


E0000 00:00:1784471693.012723     797 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


27/27 - 7s - 265ms/step - accuracy: 0.8323 - auc: 0.9235 - loss: 0.4427 - precision: 0.9325 - recall: 0.8007 - val_accuracy: 0.2530 - val_auc: 0.8976 - val_loss: 0.7669 - val_precision: 0.2439 - val_recall: 1.0000


Epoch 2/20


...

27/27 - 5s - 192ms/step - accuracy: 0.9610 - auc: 0.9759 - loss: 0.1611 - precision: 0.9609 - recall: 0.9801 - val_accuracy: 0.9277 - val_auc: 0.9909 - val_loss: 0.1375 - val_precision: 0.7917 - val_recall: 0.9500


Epoch 19/20


27/27 - 5s - 193ms/step - accuracy: 0.9610 - auc: 0.9784 - loss: 0.1521 - precision: 0.9577 - recall: 0.9837 - val_accuracy: 0.9277 - val_auc: 0.9905 - val_loss: 0.1498 - val_precision: 0.7917 - val_recall: 0.9500


Epoch 20/20


27/27 - 5s - 189ms/step - accuracy: 0.9504 - auc: 0.9776 - loss: 0.1628 - precision: 0.9554 - recall: 0.9692 - val_accuracy: 0.9398 - val_auc: 0.9944 - val_loss: 0.1230 - val_precision: 0.8261 - val_recall: 0.9500


In [5]:
probs = model.predict(X_test_in, verbose=0).ravel()
preds = (probs >= 0.5).astype(int)
auc = roc_auc_score(y_test, probs)
precision = precision_score(y_test, preds, zero_division=0)
recall = recall_score(y_test, preds, zero_division=0)
f1 = f1_score(y_test, preds, zero_division=0)
cm = confusion_matrix(y_test, preds)

print("=== Phase 5: matched negatives (LRGs, no confirmed lensing) ===")
print(f"ROC-AUC:   {auc:.3f}")
print(f"Precision: {precision:.3f}  Recall: {recall:.3f}  F1: {f1:.3f}")
print(f"Test set: n={len(y_test)}, positives={int(y_test.sum())}")
print("Confusion matrix [[TN,FP],[FN,TP]]:\n", cm)


=== Phase 5: matched negatives (LRGs, no confirmed lensing) ===
ROC-AUC:   0.956
Precision: 0.762  Recall: 0.842  F1: 0.800
Test set: n=83, positives=19
Confusion matrix [[TN,FP],[FN,TP]]:
 [[59  5]
 [ 3 16]]


## 3. Results & Discussion

| Phase | Negative sampling | ROC-AUC | Concentration-only AUC | Flux-only AUC |
|---|---|---|---|---|
| Phase 4a | Random sky positions | 0.989 | ~1.0 (trivial) | -- |
| Phase 4b | Non-blank random galaxies | 0.958 | ~1.0 (trivial) | -- |
| **Phase 5** | Matched LRGs, SLACS positions excluded | 0.956 | 0.32 (fixed) | 0.79 (partial confound remains) |

**What actually improved:** the Phase 4 confound (positives are always a
galaxy, negatives are often blank sky or any random object) is genuinely
fixed. Concentration -- which perfectly separated the classes in Phase 4 --
carries almost no signal on its own now (AUC 0.32). That's real progress,
not just a different number.

**What's still not clean:** total flux alone still gets AUC ~0.79. The
matched LRG catalog wasn't selected to match SLACS's specific redshift and
velocity-dispersion cuts, only the broad "massive elliptical" category --
so there's a residual size/brightness mismatch, most likely from redshift
distribution differences. The model's actual AUC clearly uses more than
just this one shortcut (it exceeds 0.79), but we can't cleanly attribute the
gap between 0.79 and the model's score to genuine arc detection versus some
other combination of simple photometric features.

**Read as a whole:** Phase 5 is a real, demonstrated improvement in
methodology over Phase 4, and this is about as far as this project can
honestly push real-data lens detection without a properly redshift/velocity-
dispersion-matched negative sample. Building that -- the actual SLACS parent
sample (the full spectroscopic sample SLACS drew candidates from *before*
the lensing search, not just "similar-looking galaxies") -- is the correct
next step, but requires a more specific query than a general LRG catalog
search provides. Flagging that as the honest open item rather than as
something this phase solved.
